In [ ]:
!pip install -q librosa soundfile scikit-learn pandas numpy torch
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
import zipfile, os

print('Upload your EATD-Corpus ZIP file...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

os.makedirs('/content/EATD-Corpus', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/EATD-Corpus')

print('Extracted. Folder contents:')
for root, dirs, files_ in os.walk('/content/EATD-Corpus'):
    level = root.replace('/content/EATD-Corpus', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files_[:5]:
            print(f'{subindent}{f}')

In [ ]:
import os, shutil, random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path

SAMPLE_RATE   = 16000
N_MELS        = 128
MAX_AU_FRAMES = 300
SDS_THRESHOLD = 53.0
AU_COLS = ['AU01_r','AU02_r','AU04_r','AU05_r','AU06_r','AU07_r',
           'AU09_r','AU10_r','AU12_r','AU14_r','AU15_r','AU17_r',
           'AU20_r','AU23_r','AU25_r','AU26_r','AU45_r']

EATD_ROOT   = Path('/content/EATD-Corpus')
OUTPUT_DIR  = Path('/content/eatd_processed')

def find_eatd_root(base):
    for root, dirs, files in os.walk(str(base)):
        p = Path(root)
        children = [d for d in p.iterdir() if d.is_dir()]
        names = [d.name.lower() for d in children]
        if any('train' in n for n in names):
            return p
        if any(d.name.startswith('t_') or d.name.startswith('v_') for d in children):
            return p
    return base

def read_sds(p_dir):
    for fname in ['new_label.txt', 'label.txt']:
        f = p_dir / fname
        if f.exists():
            for tok in f.read_text(errors='ignore').replace('\n',' ').split():
                try: return float(tok)
                except: pass
    return None

def extract_audio_features(wav_paths):
    combined = np.array([], dtype=np.float32)
    for wp in wav_paths:
        try:
            a, _ = librosa.load(str(wp), sr=SAMPLE_RATE)
            combined = np.concatenate([combined, a, np.zeros(int(SAMPLE_RATE*0.3), dtype=np.float32)])
        except: pass
    if len(combined) == 0:
        combined = np.zeros(SAMPLE_RATE*5, dtype=np.float32)
    feats = {}
    mfccs = librosa.feature.mfcc(y=combined, sr=SAMPLE_RATE, n_mfcc=13)
    feats['mfcc_mean'] = np.mean(mfccs, axis=1)
    feats['mfcc_std']  = np.std(mfccs, axis=1)
    rms = librosa.feature.rms(y=combined)[0]
    feats['energy_mean'] = float(np.mean(rms))
    feats['energy_std']  = float(np.std(rms))
    zcr = librosa.feature.zero_crossing_rate(combined)[0]
    feats['zcr_mean'] = float(np.mean(zcr))
    feats['zcr_std']  = float(np.std(zcr))
    sc = librosa.feature.spectral_centroid(y=combined, sr=SAMPLE_RATE)[0]
    feats['spectral_centroid_mean'] = float(np.mean(sc))
    feats['spectral_centroid_std']  = float(np.std(sc))
    return feats

def process_split(participant_dirs, split_name):
    audio_dir  = OUTPUT_DIR/'audio_features'
    text_dir   = OUTPUT_DIR/'text_features'
    facial_dir = OUTPUT_DIR/'facial_features'
    for d in [audio_dir, text_dir, facial_dir]:
        d.mkdir(parents=True, exist_ok=True)

    rows, skipped = [], 0
    for p_dir in participant_dirs:
        pid = p_dir.name
        sds = read_sds(p_dir)
        if sds is None:
            skipped += 1; continue

        binary = 1 if sds >= SDS_THRESHOLD else 0
        phq8   = round(min(24.0, max(0.0, (sds/80.0)*24.0)), 1)

        wavs = []
        for em in ['positive','negative','neutral']:
            for suf in [f'{em}_out.wav', f'{em}.wav']:
                w = p_dir/suf
                if w.exists(): wavs.append(w); break
        if wavs:
            feats = extract_audio_features(wavs)
            if feats:
                np.save(str(audio_dir/f'{pid}.npy'), feats)

        utts = []
        for em in ['positive','negative','neutral']:
            tf = p_dir/f'{em}.txt'
            if tf.exists():
                txt = tf.read_text(errors='ignore').strip()
                if txt: utts.append({'speaker':'Participant','value':txt,'text_cleaned':txt,'emotion':em})
        pd.DataFrame(utts if utts else [{'speaker':'Participant','value':'','text_cleaned':''}]
                    ).to_csv(str(text_dir/f'{pid}.csv'), index=False)

        rows_au = [{'timestamp': round(i/30.0,3), **{au:0.0 for au in AU_COLS}} for i in range(MAX_AU_FRAMES)]
        pd.DataFrame(rows_au).to_csv(str(facial_dir/f'{pid}.csv'), index=False)

        rows.append({'Participant_ID':pid,'PHQ8_Score':phq8,'PHQ8_Binary':binary,'Gender':'Unknown'})

    csv_path = OUTPUT_DIR/f'{split_name}_split_Depression_AVEC2017.csv'
    pd.DataFrame(rows).to_csv(str(csv_path), index=False)
    n_dep = sum(1 for r in rows if r['PHQ8_Binary']==1)
    print(f'  {split_name}: {len(rows)} participants | depressed={n_dep} | non-depressed={len(rows)-n_dep} | skipped={skipped}')
    return rows

eatd_root = find_eatd_root(EATD_ROOT)
print(f'EATD root detected: {eatd_root}')

train_dir = None
for name in ['train','Train','new_train']:
    if (eatd_root/name).exists():
        train_dir = eatd_root/name; break

if train_dir:
    val_dir = None
    for name in ['validation','val','dev','test','new_test']:
        if (eatd_root/name).exists():
            val_dir = eatd_root/name; break
    train_pdirs = sorted([p for p in train_dir.iterdir() if p.is_dir()])
    val_pdirs   = sorted([p for p in (val_dir or train_dir).iterdir() if p.is_dir()])
else:
    all_pdirs = sorted([p for p in eatd_root.iterdir() if p.is_dir()])
    split_idx = int(len(all_pdirs)*0.8)
    train_pdirs = all_pdirs[:split_idx]
    val_pdirs   = all_pdirs[split_idx:]
    print(f'Flat layout detected -- splitting {len(train_pdirs)} train / {len(val_pdirs)} val')

print('Processing splits...')
process_split(train_pdirs, 'train')
process_split(val_pdirs,   'dev')
shutil.copy(str(OUTPUT_DIR/'dev_split_Depression_AVEC2017.csv'),
            str(OUTPUT_DIR/'test_split_Depression_AVEC2017.csv'))
print('Preprocessing complete!')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, classification_report

N_MELS=128; MAX_AU_FRAMES=300; N_AU=17; TFIDF_DIM=500; EMBED_DIM=256; DROPOUT=0.3

class AudioEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.AdaptiveAvgPool2d((4,4)),
        )
        self.proj = nn.Sequential(
            nn.Flatten(), nn.Linear(256*4*4,512), nn.ReLU(True),
            nn.Dropout(DROPOUT), nn.Linear(512,EMBED_DIM)
        )
    def forward(self, x): return self.proj(self.cnn(x))

class TextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(TFIDF_DIM,512), nn.ReLU(True), nn.Dropout(DROPOUT),
            nn.Linear(512,EMBED_DIM)
        )
    def forward(self, x): return self.proj(x)

class FacialEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.bilstm = nn.LSTM(N_AU,128,num_layers=2,batch_first=True,
                              bidirectional=True,dropout=DROPOUT)
        self.proj = nn.Sequential(nn.Linear(256,EMBED_DIM), nn.ReLU(True), nn.Dropout(DROPOUT))
    def forward(self, x):
        _, (h,_) = self.bilstm(x)
        return self.proj(torch.cat([h[-2],h[-1]],dim=-1))

class CrossModalFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.w_a = nn.Linear(EMBED_DIM,1)
        self.w_t = nn.Linear(EMBED_DIM,1)
        self.w_f = nn.Linear(EMBED_DIM,1)
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.mlp  = nn.Sequential(nn.Linear(EMBED_DIM,EMBED_DIM), nn.ReLU(True), nn.Dropout(DROPOUT))
    def forward(self, a, t, f):
        ws = F.softmax(torch.stack([self.w_a(a),self.w_t(t),self.w_f(f)],dim=1),dim=1)
        return self.mlp(self.norm((ws * torch.stack([a,t,f],dim=1)).sum(dim=1)))

class MultimodalModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.audio  = AudioEncoder()
        self.text   = TextEncoder()
        self.facial = FacialEncoder()
        self.fusion = CrossModalFusion()
        self.head   = nn.Sequential(
            nn.Linear(EMBED_DIM,128), nn.ReLU(True), nn.Dropout(DROPOUT), nn.Linear(128,2)
        )
    def forward(self, audio, text_vec, facial):
        return self.head(self.fusion(self.audio(audio), self.text(text_vec), self.facial(facial)))

m = MultimodalModel()
print(f'Model defined. Trainable params: {sum(p.numel() for p in m.parameters() if p.requires_grad):,}')

In [ ]:
AU_COLS = ['AU01_r','AU02_r','AU04_r','AU05_r','AU06_r','AU07_r',
           'AU09_r','AU10_r','AU12_r','AU14_r','AU15_r','AU17_r',
           'AU20_r','AU23_r','AU25_r','AU26_r','AU45_r']

def load_audio_npy(path):
    data = np.load(str(path), allow_pickle=True)
    if isinstance(data, np.ndarray) and data.dtype == object: data = data.item()
    if isinstance(data, dict):
        vals = np.concatenate([np.atleast_1d(np.asarray(v,dtype=np.float32)).flatten() for v in data.values()])
        mel  = np.tile(vals[:N_MELS],(N_MELS,1)).astype(np.float32)
        mel  = (mel-mel.min())/(mel.max()-mel.min()+1e-8)
    else:
        mel = data.astype(np.float32)
        if mel.ndim==1: mel = np.expand_dims(mel,0).repeat(N_MELS,axis=0)
    T=128
    return mel[:,:T] if mel.shape[1]>=T else np.pad(mel,((0,0),(0,T-mel.shape[1])))

def load_text_raw(path):
    try:
        df = pd.read_csv(str(path))
        col = 'text_cleaned' if 'text_cleaned' in df.columns else 'value'
        return ' '.join(df[col].dropna().astype(str).tolist()).strip() or '[empty]'
    except: return '[empty]'

def load_facial(path):
    try:
        df = pd.read_csv(str(path)); df.columns = df.columns.str.strip()
        avail = [c for c in AU_COLS if c in df.columns]
        if not avail: return np.zeros((MAX_AU_FRAMES,N_AU),dtype=np.float32)
        arr = df[avail].values.astype(np.float32)
        arr = arr[:MAX_AU_FRAMES] if len(arr)>=MAX_AU_FRAMES else np.concatenate([arr,np.zeros((MAX_AU_FRAMES-len(arr),len(avail)),dtype=np.float32)])
        if arr.shape[1]<N_AU: arr=np.concatenate([arr,np.zeros((MAX_AU_FRAMES,N_AU-arr.shape[1]),dtype=np.float32)],axis=1)
        return arr
    except: return np.zeros((MAX_AU_FRAMES,N_AU),dtype=np.float32)

class EATDDataset(Dataset):
    def __init__(self, labels_csv, proc_dir, tfidf, augment=False):
        self.pdir=Path(proc_dir); self.tfidf=tfidf; self.augment=augment
        df=pd.read_csv(labels_csv); df.columns=df.columns.str.strip()
        self.labels={str(r['Participant_ID']).strip():int(r['PHQ8_Binary']) for _,r in df.iterrows()}
        self.pids=[pid for pid in self.labels if
                   (self.pdir/'audio_features'/f'{pid}.npy').exists() or
                   (self.pdir/'text_features'/f'{pid}.csv').exists()]
        print(f'  {Path(labels_csv).name}: {len(self.pids)} participants')

    def __len__(self): return len(self.pids)

    def __getitem__(self, idx):
        pid=self.pids[idx]
        ap=self.pdir/'audio_features'/f'{pid}.npy'
        mel=load_audio_npy(ap) if ap.exists() else np.zeros((N_MELS,128),dtype=np.float32)
        if self.augment:
            f0=random.randint(0,max(1,N_MELS-10)); mel[f0:min(f0+random.randint(1,10),N_MELS),:]=0
            t0=random.randint(0,max(1,128-15));    mel[:,t0:min(t0+random.randint(1,15),128)]=0
        audio_t=torch.tensor(mel,dtype=torch.float32).unsqueeze(0)
        tp=self.pdir/'text_features'/f'{pid}.csv'
        raw=load_text_raw(tp) if tp.exists() else '[empty]'
        text_t=torch.tensor(self.tfidf.transform([raw]).toarray().astype(np.float32)[0])
        fp=self.pdir/'facial_features'/f'{pid}.csv'
        facial_t=torch.tensor(load_facial(fp) if fp.exists() else np.zeros((MAX_AU_FRAMES,N_AU),dtype=np.float32))
        return audio_t, text_t, facial_t, torch.tensor(self.labels[pid],dtype=torch.long)

print('Dataset classes defined.')

In [ ]:
import time, json
from sklearn.metrics import classification_report

PROC    = '/content/eatd_processed'
OUT_DIR = Path('/content/checkpoints')
OUT_DIR.mkdir(exist_ok=True)

EPOCHS     = 30
BATCH_SIZE = 16
LR         = 3e-4
SEED       = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('Building TF-IDF...')
all_texts=[load_text_raw(f) for f in Path(PROC+'/text_features').glob('*.csv')]
tfidf=TfidfVectorizer(max_features=TFIDF_DIM, sublinear_tf=True, ngram_range=(1,2), min_df=1)
tfidf.fit(all_texts)
print(f'TF-IDF vocab: {len(tfidf.vocabulary_)}')

train_ds = EATDDataset(PROC+'/train_split_Depression_AVEC2017.csv', PROC, tfidf, augment=True)
dev_ds   = EATDDataset(PROC+'/dev_split_Depression_AVEC2017.csv',   PROC, tfidf, augment=False)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, drop_last=True)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

df_tr = pd.read_csv(PROC+'/train_split_Depression_AVEC2017.csv')
n0=int((df_tr['PHQ8_Binary']==0).sum()); n1=int((df_tr['PHQ8_Binary']==1).sum()); total=n0+n1
w0=round(total/(2*max(n0,1)),3); w1=round(total/(2*max(n1,1)),3)
print(f'Class weights: non-depressed={w0}  depressed={w1}  (n0={n0}, n1={n1})')
criterion = nn.CrossEntropyLoss(weight=torch.tensor([w0,w1],dtype=torch.float32).to(device))

model = MultimodalModel().to(device)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
opt       = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

best_f1=0.0; history={'train':[],'dev':[]}

print(f'Training {EPOCHS} epochs on {device}...')
print('='*90)

for epoch in range(EPOCHS):
    t0=time.time()
    model.train(); tloss=0; tpreds=[]; tlabels=[]
    for audio,text,facial,label in train_loader:
        audio,text,facial,label=audio.to(device),text.to(device),facial.to(device),label.to(device)
        opt.zero_grad()
        out=model(audio,text,facial)
        loss=criterion(out,label)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()
        tloss+=loss.item(); tpreds+=out.argmax(1).cpu().tolist(); tlabels+=label.cpu().tolist()
    tr_loss=tloss/len(train_loader)
    tr_acc=accuracy_score(tlabels,tpreds); tr_f1=f1_score(tlabels,tpreds,zero_division=0)

    model.eval(); dloss=0; dpreds=[]; dlabels=[]
    with torch.no_grad():
        for audio,text,facial,label in dev_loader:
            audio,text,facial,label=audio.to(device),text.to(device),facial.to(device),label.to(device)
            out=model(audio,text,facial)
            dloss+=criterion(out,label).item()
            dpreds+=out.argmax(1).cpu().tolist(); dlabels+=label.cpu().tolist()
    dv_loss=dloss/len(dev_loader)
    dv_acc=accuracy_score(dlabels,dpreds); dv_f1=f1_score(dlabels,dpreds,zero_division=0)
    dv_pr=precision_score(dlabels,dpreds,zero_division=0); dv_rc=recall_score(dlabels,dpreds,zero_division=0)
    scheduler.step()

    history['train'].append({'epoch':epoch,'loss':tr_loss,'accuracy':tr_acc,'f1':tr_f1})
    history['dev'].append(  {'epoch':epoch,'loss':dv_loss,'accuracy':dv_acc,'f1':dv_f1,'precision':dv_pr,'recall':dv_rc})

    print(f'Epoch {epoch+1:03d}/{EPOCHS} | '
          f'Train loss={tr_loss:.4f} acc={tr_acc:.3f} f1={tr_f1:.3f} | '
          f'Dev   loss={dv_loss:.4f} acc={dv_acc:.3f} f1={dv_f1:.3f} pr={dv_pr:.3f} rc={dv_rc:.3f} | '
          f'{time.time()-t0:.1f}s')

    ckpt={'epoch':epoch,'model_state':model.state_dict(),'optimizer_state':opt.state_dict(),
          'best_f1':best_f1,'history':history,
          'tfidf_vocab':tfidf.vocabulary_,'tfidf_idf':tfidf.idf_.tolist(),'args':{'task':'classification'}}
    torch.save(ckpt, OUT_DIR/'last.pt')

    if dv_f1>best_f1:
        best_f1=dv_f1
        torch.save(ckpt, OUT_DIR/'best.pt')
        print(f'  *** NEW BEST F1={best_f1:.4f} -- best.pt saved ***')

print('='*90)
print('Final Classification Report (Dev Set):')
print(classification_report(dlabels,dpreds,target_names=['Non-Depressed','Depressed']))

final_m={'accuracy':dv_acc,'f1':dv_f1,'precision':dv_pr,'recall':dv_rc,'best_f1':best_f1}
with open(OUT_DIR/'final_metrics.json','w') as f: json.dump(final_m,f,indent=2)
with open(OUT_DIR/'history.json','w') as f: json.dump(history,f,indent=2)
print(f'Done! Best F1={best_f1:.4f}  Checkpoints at /content/checkpoints/')

In [ ]:
import matplotlib.pyplot as plt

epochs_list  = [h['epoch']+1 for h in history['train']]
tr_loss_list = [h['loss'] for h in history['train']]
dv_loss_list = [h['loss'] for h in history['dev']]
tr_f1_list   = [h['f1']   for h in history['train']]
dv_f1_list   = [h['f1']   for h in history['dev']]
dv_acc_list  = [h['accuracy'] for h in history['dev']]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Multimodal Depression Detection - Training Curves', fontsize=13)

axes[0].plot(epochs_list, tr_loss_list, 'b-o', markersize=3, label='Train Loss')
axes[0].plot(epochs_list, dv_loss_list, 'r-o', markersize=3, label='Dev Loss')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs_list, tr_f1_list, 'b-o', markersize=3, label='Train F1')
axes[1].plot(epochs_list, dv_f1_list, 'r-o', markersize=3, label='Dev F1')
axes[1].set_title('F1 Score'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True)

axes[2].plot(epochs_list, dv_acc_list, 'g-o', markersize=3, label='Dev Accuracy')
axes[2].set_title('Dev Accuracy'); axes[2].set_xlabel('Epoch'); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig('/content/checkpoints/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/checkpoints/training_curves.png')

In [ ]:
from google.colab import files
import os

print('Downloading checkpoint files...')
for fname in ['best.pt', 'last.pt', 'final_metrics.json', 'history.json', 'training_curves.png']:
    fpath = f'/content/checkpoints/{fname}'
    if os.path.exists(fpath):
        files.download(fpath)
        print(f'  Downloaded: {fname}')
    else:
        print(f'  Not found: {fname}')

In [ ]:
test_texts = [
    "I have been feeling hopeless and exhausted. I can't sleep and nothing brings me joy anymore.",
    "I am doing great today! Life feels good and I am enjoying my work.",
    "I feel very sad and lonely. I don't want to do anything. Everything feels pointless.",
    "Had a productive day, went for a walk, feeling calm and grateful."
]

best_ckpt = torch.load('/content/checkpoints/best.pt', map_location='cpu')
test_model = MultimodalModel()
test_model.load_state_dict(best_ckpt['model_state'])
test_model.eval()

print('Inference on test sentences:')
print('='*70)
with torch.no_grad():
    for text in test_texts:
        vec = torch.tensor(tfidf.transform([text]).toarray().astype(np.float32))
        audio_z  = torch.zeros(1, 1, N_MELS, 128)
        facial_z = torch.zeros(1, MAX_AU_FRAMES, N_AU)
        out = test_model(audio_z, vec, facial_z)
        prob = torch.softmax(out, dim=1)[0]
        pred = 'DEPRESSED' if prob[1]>0.5 else 'NOT DEPRESSED'
        print(f'Text: "{text[:60]}..."')
        print(f'  Prediction: {pred}  (confidence: {prob[1].item():.2%})')
        print()